# Agency and private-capital comparison review

**Status:** exploratory companion notebook (non-citable)  
**Research question:** F3 — outcomes vs. private-capital baselines; B3 — statutory  
commercialization benchmark ([docs/research-questions.md](../../docs/research-questions.md))
**Canonical computation:** `scripts/data/run_benchmark_analysis.py` and the  
`agency_private_capital_baseline_comparison` Dagster asset (group
`agency_private_capital`)
**Data as of:** each artifact's own `run_manifest.json` / comparison JSON  

Companion view over the benchmark-sensitivity outputs and the per-agency baseline
comparisons, focused on cohort comparability, weighting, and alternative benchmarks.
Exploratory-tier and non-citable.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
AREA_ID = "nanotechnology"
REPORT_DIR = REPO_ROOT / "data" / "reports" / AREA_ID
RANDOM_SEED = 20260806

import json

SCRIPTS_OUTPUT = REPO_ROOT / "data" / "scripts_output"
AGENCY = "nsf"  # lowercase agency code used by the Dagster asset's output layout
AGENCY_DIR = REPO_ROOT / "data" / "processed" / "agency_private_capital" / AGENCY

## Data contract

- **Population:** the benchmark side covers Phase II awardees subject to §638(qq)(3)
  evaluation windows; the baseline-comparison side covers one agency's Phase II cohort
  against published private-capital and small-business baselines, per the
  `agency-private-capital-comparison` spec.
- **Grain:** firm for cohort outcomes; evaluation-cell grain for sensitivity tables.
- **Inputs:** every table below is read from a manifest-carrying artifact; the
  `run_manifest.json` records source provenance and parameters and travels with any
  number taken from this page.
- **Comparability caveats:** published VC/PE baselines are different populations,
  stages, and observation windows. A gap against a baseline is a framing device,
  not an estimate of program effect. Weighting choices (per-firm vs per-dollar)
  change the comparison and must be stated.

In [ ]:
ARTIFACTS = {
    "run manifest": SCRIPTS_OUTPUT / "run_manifest.json",
    "benchmark evaluation": SCRIPTS_OUTPUT / "benchmark_evaluation.json",
    "transition-rate detail": SCRIPTS_OUTPUT / "transition_rate_detail.csv",
    "commercialization-rate detail": SCRIPTS_OUTPUT / "commercialization_rate_detail.csv",
    "agency cohort outcomes": AGENCY_DIR / "agency_cohort_outcomes.parquet",
    "agency baseline comparison": AGENCY_DIR / "agency_baseline_comparison.json",
}
GENERATORS = {
    "run manifest": "scripts/data/run_benchmark_analysis.py",
    "benchmark evaluation": "scripts/data/run_benchmark_analysis.py",
    "transition-rate detail": "scripts/data/run_benchmark_analysis.py",
    "commercialization-rate detail": "scripts/data/run_benchmark_analysis.py",
    "agency cohort outcomes": "the agency_private_capital_baseline_comparison asset",
    "agency baseline comparison": "the agency_private_capital_baseline_comparison asset",
}
pd.DataFrame(
    [
        {"artifact": name, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()}
        for name, path in ARTIFACTS.items()
    ]
)

def load_artifact(name: str) -> pd.DataFrame:
    """Read a canonical CSV artifact, or return an empty frame with a hint."""
    path = ARTIFACTS[name]
    if not path.exists():
        print(
            f"Missing {path.relative_to(REPO_ROOT)} — artifact not present; "
            f"run {GENERATORS[name]} first."
        )
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

## Provenance first

Read the run manifest before any number. If the manifest is absent, the sensitivity
tables have no declared data cut and should not be compared across runs.

In [ ]:
manifest_path = ARTIFACTS["run manifest"]
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    display(pd.json_normalize(manifest).T.rename(columns={0: "value"}))
else:
    print(
        f"Missing {manifest_path.relative_to(REPO_ROOT)} — artifact not present; "
        "run scripts/data/run_benchmark_analysis.py first."
    )

## Benchmark sensitivity views

Transition- and commercialization-rate detail under the margins the script varied.
These are diagnostics on the benchmark's stability, not program findings.

In [ ]:
transition_detail = load_artifact("transition-rate detail")
commercialization_detail = load_artifact("commercialization-rate detail")
for label, table in (
    ("transition-rate detail", transition_detail),
    ("commercialization-rate detail", commercialization_detail),
):
    if not table.empty:
        print(f"{label}: {len(table):,} rows")
        display(table.head(10))

## Agency cohort vs. published baselines

The asset's comparison JSON pairs each cohort statistic with its published baseline
and source. Alternative benchmarks belong in the baseline registry
(`packages/sbir-analytics/.../agency_private_capital/baselines`), not hand-typed here.

In [ ]:
comparison_path = ARTIFACTS["agency baseline comparison"]
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text())
    baseline_view = pd.json_normalize(comparison)
else:
    print(
        f"Missing {comparison_path.relative_to(REPO_ROOT)} — artifact not present; "
        f"materialize the agency_private_capital assets for '{AGENCY}' first."
    )
    baseline_view = pd.DataFrame()
baseline_view

In [ ]:
outcomes_path = ARTIFACTS["agency cohort outcomes"]
if outcomes_path.exists():
    outcomes = pd.read_parquet(outcomes_path)
    print(f"{len(outcomes):,} cohort firms")
    display(outcomes.head())
else:
    print(
        f"Missing {outcomes_path.relative_to(REPO_ROOT)} — artifact not present; "
        f"materialize the agency_private_capital assets for '{AGENCY}' first."
    )

## Interpretation log

| Comparison | Weighting | Population mismatch | Defensible statement |
|---|---|---|---|
| _Draft_ | _Per-firm vs per-dollar_ | _Stage/window/selection differences_ | _Framing, not effect_ |